#### Bert evaluation

##### How much did the model learn?

Load the trained model, load the tokenized data, mask some tokens, and ask: how well does the model recover them?

In [10]:
import os, json
import numpy as np
import torch
import pandas as pd
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
from transformers import BertForMaskedLM, BertConfig

In [7]:
commonpath = "/home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/"
saved_dir = os.path.join(commonpath,"models/bert_dev_learned/")

In [11]:
# loads a pretrained BERT model for masked language modeling
# config.json defines the model architecture and settings
# model.safetensors contains the trained weights
# model = BertForMaskedLM.from_pretrained(saved_dir)

meta_path = os.path.join(saved_dir, "vocab_meta.json")
with open(meta_path) as f:
    meta = json.load(f)

# Meta is a dictionary with the following keys
print('keys of meta dictionary -> ',meta.keys())

required = ["gene2id","PAD_ID","CLS_ID","SEP_ID","MASK_ID","DEV_DOM_ID","CANCER_DOM_ID","vocab_size"]
missing = [k for k in required if k not in meta]
assert not missing, f"vocab_meta.json missing keys: {missing}"

# unpacking the dictionary into variables.
gene2id       = meta["gene2id"]
PAD_ID        = meta["PAD_ID"]
CLS_ID        = meta["CLS_ID"]
SEP_ID        = meta["SEP_ID"]
MASK_ID       = meta["MASK_ID"]
DEV_DOM_ID    = meta["DEV_DOM_ID"]
CANCER_DOM_ID = meta["CANCER_DOM_ID"]
vocab_size    = meta["vocab_size"]

# Ensure special IDs are in range and distinct (no two special tokens share the same ID)
specials = [PAD_ID, CLS_ID, SEP_ID, MASK_ID, DEV_DOM_ID, CANCER_DOM_ID]
assert len(set(specials)) == len(specials), "Some special token IDs collide."
assert all(0 <= x < vocab_size for x in specials), "A special token ID is out of vocab range."

keys of meta dictionary ->  dict_keys(['gene2id', 'PAD_ID', 'CLS_ID', 'SEP_ID', 'MASK_ID', 'DEV_DOM_ID', 'CANCER_DOM_ID', 'vocab_size'])


In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
# Load trained model + data
# -------------------------
model = BertForMaskedLM.from_pretrained(saved_dir).to(device)
model.eval()

X_tokens_eval = np.load(os.path.join(saved_dir, "X_tokens.npy"))
X_attn_eval   = np.load(os.path.join(saved_dir, "X_attn.npy"))

# -------------------------
# Dataset
# -------------------------
class TokenDataset(Dataset):
    def __init__(self, tokens, attn):
        self.tokens = torch.as_tensor(tokens, dtype=torch.long)
        self.attn   = torch.as_tensor(attn, dtype=torch.long)

    def __len__(self):
        return self.tokens.shape[0]

    def __getitem__(self, i):
        return {
            "input_ids": self.tokens[i],
            "attention_mask": self.attn[i],
        }

ds_eval = TokenDataset(X_tokens_eval, X_attn_eval)


same collator i used on training

In [13]:
SPECIAL_IDS = {PAD_ID, CLS_ID, SEP_ID, MASK_ID, DEV_DOM_ID, CANCER_DOM_ID}

@dataclass
class SimpleMLMCollator:
    mlm_probability: float = 0.15

    def __call__(self, batch):
        input_ids = torch.stack([b["input_ids"] for b in batch], dim=0)
        attention_mask = torch.stack([b["attention_mask"] for b in batch], dim=0)

        labels = input_ids.clone()

        maskable = torch.ones_like(input_ids, dtype=torch.bool)
        for sid in SPECIAL_IDS:
            maskable &= input_ids != sid

        prob = torch.full(input_ids.shape, self.mlm_probability, dtype=torch.float32)
        sampled = torch.bernoulli(prob).bool() & maskable

        replace_mask = sampled & (torch.rand_like(input_ids, dtype=torch.float32) < 0.8)
        random_mask = sampled & (~replace_mask) & (torch.rand_like(input_ids, dtype=torch.float32) < 0.5)

        input_ids[replace_mask] = MASK_ID

        rand_ids = torch.randint(
            low=4,
            high=model.config.vocab_size,
            size=input_ids.shape,
            dtype=torch.long,
        )
        input_ids[random_mask] = rand_ids[random_mask]

        labels[~sampled] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

Then evaluate

In [14]:
def evaluate_mlm_topk(model, dataset, batch_size=16, topks=(1, 5, 10)):
    collator = SimpleMLMCollator(mlm_probability=0.15)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collator,
    )

    total = 0
    correct = {k: 0 for k in topks}
    losses = []

    model.eval()

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            labels = batch["labels"]

            out = model(**batch)
            logits = out.logits

            losses.append(out.loss.item())

            masked_positions = labels != -100
            true_ids = labels[masked_positions]
            pred_logits = logits[masked_positions]

            total += true_ids.numel()

            for k in topks:
                topk_ids = pred_logits.topk(k, dim=-1).indices
                correct[k] += (
                    topk_ids == true_ids.unsqueeze(-1)
                ).any(dim=-1).sum().item()

    results = {
        "mlm_loss": float(np.mean(losses)),
        "n_masked_tokens": int(total),
    }

    for k in topks:
        results[f"top{k}_accuracy"] = correct[k] / total

    return results

In [15]:
trained_metrics = evaluate_mlm_topk(
    model=model,
    dataset=ds_eval,
    batch_size=16,
    topks=(1, 5, 10),
)

trained_metrics

{'mlm_loss': 4.635438947512642,
 'n_masked_tokens': 1207067,
 'top1_accuracy': 0.1788169173707839,
 'top5_accuracy': 0.29236736651735157,
 'top10_accuracy': 0.3609393679058412}

Add a random-model baseline

In [16]:
config = model.config

random_model = BertForMaskedLM(config).to(device)
random_model.eval()

random_metrics = evaluate_mlm_topk(
    model=random_model,
    dataset=ds_eval,
    batch_size=16,
    topks=(1, 5, 10),
)

comparison = pd.DataFrame([
    {"model": "Random initialization", **random_metrics},
    {"model": "Trained model", **trained_metrics},
])

comparison

,model,mlm_loss,n_masked_tokens,top1_accuracy,top5_accuracy,top10_accuracy
0,Random initialization,9.690004,1208640,0.000061,0.000264,0.000620
1,Trained model,4.635439,1207067,0.178817,0.292367,0.360939


Repeat with different masks

In [17]:
all_results = []

for seed in range(10):
    torch.manual_seed(seed)
    np.random.seed(seed)

    metrics = evaluate_mlm_topk(
        model=model,
        dataset=ds_eval,
        batch_size=16,
        topks=(1, 5, 10),
    )

    metrics["seed"] = seed
    metrics["model"] = "Trained model"
    all_results.append(metrics)

df_trained_repeats = pd.DataFrame(all_results)

df_trained_repeats.describe()

,mlm_loss,n_masked_tokens,top1_accuracy,top5_accuracy,top10_accuracy,seed
count,10.000000,1.000000e+01,10.000000,10.000000,10.000000,10.00000
mean,4.636916,1.206635e+06,0.178434,0.291457,0.360244,4.50000
std,0.001160,1.138142e+03,0.000202,0.000347,0.000378,3.02765
min,4.634456,1.204701e+06,0.178170,0.290931,0.359672,0.00000
25%,4.636643,1.205839e+06,0.178266,0.291200,0.360072,2.25000
50%,4.636987,1.206679e+06,0.178394,0.291463,0.360268,4.50000
75%,4.637393,1.207523e+06,0.178593,0.291742,0.360555,6.75000
max,4.638704,1.208185e+06,0.178780,0.291940,0.360698,9.00000


Then for the random model

In [18]:
all_results = []

for seed in range(10):
    torch.manual_seed(seed)
    np.random.seed(seed)

    metrics = evaluate_mlm_topk(
        model=random_model,
        dataset=ds_eval,
        batch_size=16,
        topks=(1, 5, 10),
    )

    metrics["seed"] = seed
    metrics["model"] = "Random initialization"
    all_results.append(metrics)

df_random_repeats = pd.DataFrame(all_results)

df_all = pd.concat(
    [df_random_repeats, df_trained_repeats],
    ignore_index=True,
)

df_all.groupby("model")[
    ["mlm_loss", "top1_accuracy", "top5_accuracy", "top10_accuracy"]
].agg(["mean", "std"])

mlm_loss           top1_accuracy            \
                           mean       std          mean       std   
model                                                               
Random initialization  9.689852  0.000191      0.000067  0.000010   
Trained model          4.636916  0.001160      0.178434  0.000202   

                      top5_accuracy           top10_accuracy            
                               mean       std           mean       std  
model                                                                   
Random initialization      0.000265  0.000009       0.000632  0.000015  
Trained model              0.291457  0.000347       0.360244  0.000378

In [22]:
df_random_repeats

,mlm_loss,n_masked_tokens,top1_accuracy,top5_accuracy,top10_accuracy,seed,model
0,9.689842,1207757,0.000065,0.000265,0.000647,0,Random initialization
1,9.689681,1205463,0.000053,0.000258,0.000635,1,Random initialization
2,9.689621,1205719,0.000069,0.000259,0.000615,2,Random initialization
3,9.689836,1206235,0.000078,0.000264,0.000638,3,Random initialization
4,9.690298,1204701,0.000065,0.000266,0.000615,4,Random initialization
5,9.690019,1207409,0.000063,0.000272,0.000625,5,Random initialization
6,9.689843,1207561,0.000054,0.000258,0.000635,6,Random initialization
7,9.689823,1207123,0.000070,0.000253,0.000619,7,Random initialization
8,9.689831,1206200,0.000087,0.000281,0.000663,8,Random initialization
9,9.689720,1208185,0.000067,0.000273,0.000632,9,Random initialization


In [19]:
trained_metrics = evaluate_mlm_topk(model, ds_eval)
random_metrics = evaluate_mlm_topk(BertForMaskedLM(model.config).to(device), ds_eval)

In [20]:
trained_metrics

{'mlm_loss': 4.6363377350336785,
 'n_masked_tokens': 1206058,
 'top1_accuracy': 0.17841430511633768,
 'top5_accuracy': 0.2913317601641049,
 'top10_accuracy': 0.360338391685972}

In [21]:
random_metrics

{'mlm_loss': 9.6860697951762,
 'n_masked_tokens': 1209735,
 'top1_accuracy': 4.050473864110735e-05,
 'top5_accuracy': 0.00022732251278172492,
 'top10_accuracy': 0.0005125089379078889}